# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [1]:
# Write your code below.
%load_ext dotenv
%dotenv 

In [2]:
import os
price_data_path = os.getenv("PRICE_DATA")
price_data_path #the environment variable is loaded into a variable

'../../05_src/data/prices/'

In [3]:
import dask.dataframe as dd

c:\Users\carva\anaconda3\envs\dsi_participant\lib\site-packages\dask\dataframe\_pyarrow_compat.py:15: FutureWarning: Minimal version of pyarrow will soon be increased to 14.0.1. You are using 11.0.0. Please consider upgrading.
  warnings.warn(


+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [4]:
import os
from glob import glob

# Write your code below.
parquet_files = glob(os.path.join(price_data_path, '**/*.parquet'), recursive=True)

dd_px = dd.read_parquet(parquet_files).set_index("Ticker")

dd_px.head()

Price,Date,Adj Close,Close,High,Low,Open,Volume,Year
Ticker,,,,,,,,
A,2000-01-03 00:00:00+00:00,43.382847,51.502148,56.464592,48.193848,56.330471,4674353.0,2000
A,2000-01-04 00:00:00+00:00,40.068882,47.567955,49.266811,46.316166,48.730328,4765083.0,2000
A,2000-01-05 00:00:00+00:00,37.583393,44.617310,47.567955,43.141991,47.389126,5758642.0,2000
A,2000-01-06 00:00:00+00:00,36.152378,42.918453,44.349072,41.577251,44.080830,2534434.0,2000
A,2000-01-07 00:00:00+00:00,39.165066,46.494991,47.165592,42.203148,42.247852,2819626.0,2000


In [ ]:
dd_px.index.unique().compute() #Here I wanted to confirm that there are multiple tickers so I can then group them

c:\Users\carva\anaconda3\envs\dsi_participant\lib\site-packages\pandas\core\frame.py:717: DeprecationWarning: Passing a BlockManager to DataFrame is deprecated and will raise in a future version. Use public APIs instead.
  warnings.warn(
c:\Users\carva\anaconda3\envs\dsi_participant\lib\site-packages\pandas\core\frame.py:717: DeprecationWarning: Passing a BlockManager to DataFrame is deprecated and will raise in a future version. Use public APIs instead.
  warnings.warn(


c:\Users\carva\anaconda3\envs\dsi_participant\lib\site-packages\pandas\core\frame.py:717: DeprecationWarning: Passing a BlockManager to DataFrame is deprecated and will raise in a future version. Use public APIs instead.
  warnings.warn(
c:\Users\carva\anaconda3\envs\dsi_participant\lib\site-packages\pandas\core\frame.py:717: DeprecationWarning: Passing a BlockManager to DataFrame is deprecated and will raise in a future version. Use public APIs instead.
  warnings.warn(
c:\Users\carva\anaconda3\envs\dsi_participant\lib\site-packages\pyarrow\pandas_compat.py:766: DeprecationWarning: DatetimeTZBlock is deprecated and will be removed in a future version. Use public APIs instead.
  klass=_int.DatetimeTZBlock,
c:\Users\carva\anaconda3\envs\dsi_participant\lib\site-packages\pandas\core\frame.py:717: DeprecationWarning: Passing a BlockManager to DataFrame is deprecated and will raise in a future version. Use public APIs instead.
  warnings.warn(
c:\Users\carva\anaconda3\envs\dsi_participant\

Index(['BEN', 'KDP', 'NTAP', 'TT', 'DFS', 'KR', 'ETSY', 'EVRG', 'STLD', 'ALLE',
       ...
       'TXT', 'DOV', 'ED', 'FOX', 'IP', 'IQV', 'FRT', 'JPM', 'PLD', 'VTR'],
      dtype='object', name='Ticker', length=503)

For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [7]:
# Write your code below.
#Adding the lags for the two columns
dd_shift = dd_px.groupby("Ticker", group_keys=False).apply(
    lambda x: x.assign(
        Close_lag_1=x['Close'].shift(1),         # Lag for Close
        Adj_Close_lag_1=x['Adj Close'].shift(1)  # Lag for Adj Close
    )
)

C:\Users\carva\AppData\Local\Temp\ipykernel_19104\1411561393.py:3: UserWarning: `meta` is not specified, inferred from partial data. Please provide `meta` if the result is unexpected.
  Before: .apply(func)
  After:  .apply(func, meta={'x': 'f8', 'y': 'f8'}) for dataframe result
  or:     .apply(func, meta=('x', 'f8'))            for series result
  dd_shift = dd_px.groupby("Ticker", group_keys=False).apply(


In [8]:
#Adding the return columns
dd_rets = dd_shift.assign(
    Returns=lambda x: x['Close'] / x['Close_lag_1'] - 1  # Returns calculation
)

In [9]:
#Adding the hi_lo_range col and assigning it to a new df
dd_feat = dd_rets.assign(
    hi_lo_range=lambda x: x['High'] - x['Low']  # High minus Low
)

+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [17]:
# Write your code below.
# Converting the Dask df to a Pandas df
pandas_df = dd_feat.compute()


In [33]:
pandas_df['returns_ma_10'] = pandas_df.groupby(level=0)['Returns'].transform(
    lambda x: x.rolling(10).mean()
)

pandas_df.head(12)

Price,Date,Adj Close,Close,High,Low,Open,Volume,Year,Close_lag_1,Adj_Close_lag_1,Returns,hi_lo_range,returns_ma_10
Ticker,,,,,,,,,,,,,
DOV,2016-01-04 00:00:00+00:00,42.421429,49.789986,49.806141,48.327950,48.562199,2428956.0,2016,NaN,NaN,NaN,1.478191,NaN
DOV,2016-01-05 00:00:00+00:00,41.616238,48.844910,49.983845,48.441032,49.806141,1335059.0,2016,49.789986,42.421429,-0.018981,1.542812,NaN
DOV,2016-01-06 00:00:00+00:00,40.735313,47.810986,48.505653,47.576736,48.012924,1532025.0,2016,48.844910,41.616238,-0.021167,0.928917,NaN
DOV,2016-01-07 00:00:00+00:00,40.150345,47.124393,47.786755,46.631664,46.930534,2011131.0,2016,47.810986,40.735313,-0.014361,1.155090,NaN
DOV,2016-01-08 00:00:00+00:00,39.806240,46.720516,47.528271,46.550888,47.302101,2681508.0,2016,47.124393,40.150345,-0.008570,0.977383,NaN
DOV,2016-01-11 00:00:00+00:00,39.207485,46.017773,47.075928,45.678513,46.849758,2736104.0,2016,46.720516,39.806240,-0.015041,1.397415,NaN
DOV,2016-01-12 00:00:00+00:00,38.863377,45.613892,47.035542,44.491116,46.809368,5218046.0,2016,46.017773,39.207485,-0.008777,2.544426,NaN
DOV,2016-01-13 00:00:00+00:00,38.071926,44.684975,46.453957,44.612278,45.008080,2506455.0,2016,45.613892,38.863377,-0.020365,1.841679,NaN
DOV,2016-01-14 00:00:00+00:00,38.340340,45.000000,45.573505,43.812599,44.733440,3856865.0,2016,44.684975,38.071926,0.007050,1.760906,NaN


Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?  
It was not necesary to convert it to Pandas because Dask can also handle the same operations.
+ Would it have been better to do it in Dask? Why?  
It would've been better because we are using a large dataset so the entire dataset is not loaded in memory.

(1 pt)

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.